# Shared Pipeline Utilities — Logging, Quarantine, Incremental Merge
Reusable functions (log_event, quarantine_records, merge_upsert) imported by
every other notebook via %run. Centralizes logging and data-quality handling
so it's consistent across all pipeline stages instead of duplicated.

In [0]:
catalog = "workspace"
import uuid
import time
from pyspark.sql.functions import lit, current_timestamp, col

RUN_ID = str(uuid.uuid4())
ops_schema = f"{catalog}.retail_ops"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ops_schema}")
LOG_TABLE = f"{ops_schema}.pipeline_logs"

print("Pipeline utils initializing. RUN_ID =", RUN_ID, "| ops schema =", ops_schema)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, LongType

LOG_SCHEMA = StructType([
    StructField("run_id", StringType(), True),
    StructField("layer", StringType(), True),
    StructField("task_name", StringType(), True),
    StructField("status", StringType(), True),
    StructField("message", StringType(), True),
    StructField("rows_affected", LongType(), True),
    StructField("log_timestamp", StringType(), True),
])

def log_event(layer, task_name, status, message="", rows_affected=None):
    row = [(
        RUN_ID, layer, task_name, status, str(message)[:2000],
        int(rows_affected) if rows_affected is not None else None,
        time.strftime("%Y-%m-%d %H:%M:%S")
    )]
    log_df = (spark.createDataFrame(row, schema=LOG_SCHEMA)
              .withColumn("log_timestamp", col("log_timestamp").cast("timestamp")))
    if not spark.catalog.tableExists(LOG_TABLE):
        log_df.write.format("delta").mode("overwrite").saveAsTable(LOG_TABLE)
    else:
        log_df.write.format("delta").mode("append").saveAsTable(LOG_TABLE)
    print(f"[{status}] ({layer}) {task_name}: {message}")

In [0]:
def quarantine_records(df, is_valid_condition, quarantine_table_name, reason):
    clean_df = df.filter(is_valid_condition)
    bad_df = df.filter(~is_valid_condition)
    bad_count = bad_df.count()

    if bad_count > 0:
        target = f"{ops_schema}.{quarantine_table_name}_quarantine"
        tagged = (bad_df
                  .withColumn("_quarantine_reason", lit(reason))
                  .withColumn("_quarantine_run_id", lit(RUN_ID))
                  .withColumn("_quarantine_timestamp", current_timestamp()))
        if not spark.catalog.tableExists(target):
            tagged.write.format("delta").mode("overwrite").saveAsTable(target)
        else:
            tagged.write.format("delta").mode("append").saveAsTable(target)
        log_event("quarantine", quarantine_table_name, "WARN", f"{bad_count} rows quarantined: {reason}", bad_count)
    else:
        log_event("quarantine", quarantine_table_name, "SUCCESS", f"No rows failed ({reason})", 0)

    return clean_df

print("log_event() and quarantine_records() are ready to use.")

In [0]:
log_event("test", "manual_check", "SUCCESS", "First test of the logging system", 0)
display(spark.sql(f"SELECT * FROM {LOG_TABLE} ORDER BY log_timestamp DESC"))

In [0]:
def merge_upsert(df, target_table_fq, join_keys):
    """
    Generic incremental upsert: creates the table if it doesn't exist,
    otherwise only inserts new rows and updates changed ones — matching rows
    that haven't changed are left untouched.
    """
    from delta.tables import DeltaTable

    if not spark.catalog.tableExists(target_table_fq):
        df.write.format("delta").mode("overwrite").saveAsTable(target_table_fq)
        row_count = df.count()
        print(f"{target_table_fq}: created with {row_count} rows")
        return row_count

    target_delta = DeltaTable.forName(spark, target_table_fq)
    condition = " AND ".join([f"target.{k} = source.{k}" for k in join_keys])
    (target_delta.alias("target")
     .merge(df.alias("source"), condition)
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute())
    row_count = spark.table(target_table_fq).count()
    print(f"{target_table_fq}: merged, now {row_count} rows total")
    return row_count